# ML Interview Prep: Math & Statistics

**Based on Chip Huyen's ML Interviews Book -- Chapter 5: Math**

Source: https://huyenchip.com/ml-interviews-book/

This notebook covers core math and statistics concepts tested in ML interviews at top tech companies. Each section includes theory, intuition, and working code.

In [ ]:
import numpy as np
import scipy.stats as stats
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print('Libraries loaded successfully')

---
## SECTION 1: Vectors & Matrices

### Why This Matters
Linear algebra is the backbone of ML. Neural networks are matrix multiplications. PCA uses eigendecomposition. Attention mechanisms use dot products. You will be asked about these in every ML interview.

### 1.1 Dot Product and Cosine Similarity

**Interview Question:** How does cosine similarity differ from dot product? When would you use each?

**Answer:** The dot product measures the projection of one vector onto another, but is sensitive to vector magnitude. Cosine similarity normalizes by magnitude, measuring only the angle. In NLP, two documents with the same topics but different lengths should have high cosine similarity -- the dot product would penalize the shorter document unfairly.

In [ ]:
a = np.array([1.0, 2.0, 3.0])
b = np.array([4.0, 5.0, 6.0])

dot_product = np.dot(a, b)
cosine_sim = np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print(f"Dot product: {dot_product:.4f}")
print(f"Cosine similarity: {cosine_sim:.4f}")
print(f"Angle between vectors: {np.degrees(np.arccos(cosine_sim)):.2f} degrees")

# NLP example: scaled vs unscaled document vectors
doc1 = np.array([1.0, 1.0, 0.0])    # mentions python and ml
doc2 = np.array([10.0, 10.0, 0.0])  # same topics, 10x longer
doc3 = np.array([0.0, 1.0, 1.0])    # mentions ml and data

print("\nNLP document vectors:")
print(f"Dot(doc1,doc2): {np.dot(doc1,doc2):.1f}  <- inflated by length")
cs12 = np.dot(doc1,doc2)/(np.linalg.norm(doc1)*np.linalg.norm(doc2))
cs13 = np.dot(doc1,doc3)/(np.linalg.norm(doc1)*np.linalg.norm(doc3))
print(f"Cosine(doc1,doc2): {cs12:.4f}  <- identical topics")
print(f"Cosine(doc1,doc3): {cs13:.4f}  <- partially similar")

### 1.2 Matrix Multiplication Complexity

**Interview Question:** What is the time complexity of multiplying matrices of shapes (m x k) and (k x n)?

**Answer:** O(m * k * n). In deep learning this dominates compute for large hidden dimensions. This motivates low-rank approximations like LoRA for fine-tuning.

In [ ]:
import time

sizes = [50, 100, 200, 400]
times = []
print('Matrix multiplication timing (O(n^3) for square matrices):')
for n in sizes:
    A = np.random.randn(n, n)
    B = np.random.randn(n, n)
    start = time.perf_counter()
    C = A @ B
    elapsed = time.perf_counter() - start
    times.append(elapsed)
    print(f"n={n:4d}: time={elapsed*1000:.2f}ms")

if times[0] > 1e-9:
    print(f"\nTime ratio (n=400 vs n=50): {times[-1]/times[0]:.1f}x")
print('Theoretical: (400/50)^3 = 512x more operations')

### 1.3 Transpose, Inverse, Determinant

**Key Facts:**
- (AB)^T = B^T A^T
- (AB)^-1 = B^-1 A^-1
- det(AB) = det(A) * det(B)
- A matrix is invertible iff det != 0
- Normal equation: theta = (X^T X)^-1 X^T y requires X^T X to be invertible

In [ ]:
A = np.array([[2.0, 1.0], [5.0, 3.0]])
B = np.array([[1.0, 2.0], [3.0, 4.0]])

print('Matrix A:')
print(A)
print('\nTranspose A^T:')
print(A.T)

print(f"\ndet(A) = {np.linalg.det(A):.4f}")
A_inv = np.linalg.inv(A)
print('\nInverse A^-1:')
print(A_inv)
print('\nA @ A^-1 (should be Identity):')
print(np.round(A @ A_inv, 6))

print(f"\ndet(A)*det(B) = {np.linalg.det(A)*np.linalg.det(B):.4f}")
print(f"det(A @ B)    = {np.linalg.det(A @ B):.4f}")

singular = np.array([[1.0, 2.0], [2.0, 4.0]])
print(f"\nSingular matrix det = {np.linalg.det(singular):.8f}")
print('det~=0 means rows are linearly dependent, matrix not invertible')

### 1.4 Eigenvalues and Eigenvectors

**Interview Question:** What do eigenvalues and eigenvectors represent geometrically? How are they used in PCA?

**Answer:** An eigenvector is a direction that only scales (not rotates) when the matrix is applied. In PCA, eigenvectors of the covariance matrix are the principal components. Eigenvalues quantify variance explained by each component.

**Formula:** A*v = lambda * v

In [ ]:
np.random.seed(42)
X = np.random.multivariate_normal([0, 0], [[3, 2], [2, 2]], size=300)
cov_matrix = np.cov(X.T)
eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)

# Sort descending by eigenvalue
idx = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]

print('Covariance matrix:')
print(np.round(cov_matrix, 4))
print(f"\nEigenvalues: {eigenvalues.round(4)}")
print(f"Eigenvectors:\n{eigenvectors.round(4)}")
print(f"\nVariance explained by PC1: {eigenvalues[0]/eigenvalues.sum()*100:.1f}%")
print(f"Variance explained by PC2: {eigenvalues[1]/eigenvalues.sum()*100:.1f}%")

# Verify Av = lambda*v
v1 = eigenvectors[:, 0]
Av = cov_matrix @ v1
lv = eigenvalues[0] * v1
print(f"\nAv       = {Av.round(6)}")
print(f"lambda*v = {lv.round(6)}")
print(f"Match: {np.allclose(Av, lv)}")

### 1.5 SVD -- Singular Value Decomposition

**Interview Question:** Explain SVD. How is it used in ML?

**Answer:** A = U * Sigma * V^T where U, V are orthogonal and Sigma is diagonal with singular values (all >= 0).

**Applications:**
- PCA / dimensionality reduction: keep top-k singular vectors
- Recommender systems: latent factor models
- Image compression: truncated SVD
- NLP: Latent Semantic Analysis (LSA)
- Pseudo-inverse: solving least-squares problems numerically

In [ ]:
# SVD for image compression
np.random.seed(0)
true_pattern = np.outer(
    np.sin(np.linspace(0, np.pi, 50)),
    np.cos(np.linspace(0, np.pi, 50))
)
image = true_pattern + 0.3 * np.random.randn(50, 50)

U, sigma, Vt = np.linalg.svd(image)
print(f"Image shape: {image.shape}")
print(f"Top 10 singular values: {sigma[:10].round(3)}")

def reconstruct(U, sigma, Vt, k):
    return U[:, :k] @ np.diag(sigma[:k]) @ Vt[:k, :]

total_energy = np.sum(sigma**2)
print('\nk   Frob_Error  Compression  Energy%')
for k in [1, 5, 10, 25, 50]:
    rec = reconstruct(U, sigma, Vt, k)
    err = np.linalg.norm(image - rec, 'fro')
    orig = image.shape[0] * image.shape[1]
    comp = k * (image.shape[0] + image.shape[1] + 1)
    energy = np.sum(sigma[:k]**2) / total_energy * 100
    print(f"{k:2d}  {err:10.4f}  {orig/comp:10.1f}x  {energy:7.1f}%")

---
## SECTION 2: Probability

*Most heavily tested section in ML interviews. Bayes theorem, distributions, and estimation appear in virtually every senior ML interview.*

### 2.1 Bayes Theorem -- Three Real Examples

**Formula:** P(A|B) = P(B|A) * P(A) / P(B)

**Components:**
- P(A) = prior probability
- P(B|A) = likelihood
- P(A|B) = posterior probability
- P(B) = evidence (normalizing constant)

**The classic interview trap:** Ignoring the base rate. Even a highly accurate test produces many false positives when the condition is rare.

In [ ]:
# Example 1: Medical Test
p_disease = 0.01
p_pos_given_disease = 0.99   # sensitivity
p_pos_given_healthy = 0.05   # false positive rate
p_healthy = 1 - p_disease

p_pos = p_pos_given_disease*p_disease + p_pos_given_healthy*p_healthy
p_disease_pos = (p_pos_given_disease * p_disease) / p_pos

print('=== Medical Test ===')
print(f"Prevalence: {p_disease*100:.1f}%")
print(f"Sensitivity: {p_pos_given_disease*100:.1f}%, FPR: {p_pos_given_healthy*100:.1f}%")
print(f"P(disease | positive test): {p_disease_pos*100:.2f}%")
print('Surprise: only ~16.7% despite 99% accurate test (low base rate!)')

# Example 2: Spam Filter
p_spam = 0.3
p_free_given_spam = 0.8
p_free_given_ham = 0.1
p_free = p_free_given_spam*p_spam + p_free_given_ham*(1-p_spam)
p_spam_free = p_free_given_spam * p_spam / p_free

print('\n=== Spam Filter ===')
print(f"P(spam | contains word free): {p_spam_free:.4f}")

# Example 3: Fraud Detection
p_fraud = 0.001
p_flag_fraud = 0.95
p_flag_legit = 0.02
p_flag = p_flag_fraud*p_fraud + p_flag_legit*(1-p_fraud)
p_fraud_flag = p_flag_fraud * p_fraud / p_flag

print('\n=== Fraud Detection ===')
print(f"P(fraud | flagged): {p_fraud_flag:.4f} = {p_fraud_flag*100:.2f}%")
print(f"Of 1000 flagged, only ~{p_fraud_flag*1000:.0f} are actual fraud")
print('High AUC does not guarantee useful precision at low fraud rates!')

### 2.2 Conditional Probability -- Monty Hall Problem

**Problem:** Three doors: one car, two goats. You pick door 1. The host (who knows) opens door 3 (a goat). Should you switch to door 2?

**Answer: YES -- switching wins 2/3 of the time.**

**Intuition:** Your initial pick had 1/3 probability. The host's action (non-random -- always reveals a goat) doesn't help your door. The remaining 2/3 probability concentrates on the other unopened door.

In [ ]:
def monty_hall_sim(n_trials, switch):
    wins = 0
    for _ in range(n_trials):
        car = np.random.randint(0, 3)
        pick = np.random.randint(0, 3)
        host_choices = [d for d in range(3) if d != pick and d != car]
        host = np.random.choice(host_choices)
        if switch:
            final = [d for d in range(3) if d != pick and d != host][0]
        else:
            final = pick
        if final == car:
            wins += 1
    return wins / n_trials

np.random.seed(0)
n = 100000
stay_rate = monty_hall_sim(n, switch=False)
switch_rate = monty_hall_sim(n, switch=True)
print(f"Monty Hall Simulation ({n:,} trials)")
print(f"Stay:   {stay_rate:.4f}  (theory: 0.3333)")
print(f"Switch: {switch_rate:.4f}  (theory: 0.6667)")
print(f"Switching is {switch_rate/stay_rate:.2f}x better")

### 2.3 Joint, Marginal, and Conditional Distributions

**Definitions:**
- Joint P(X,Y): probability of both events simultaneously
- Marginal P(X) = sum_y P(X,Y=y): sum/integrate out the other variable
- Conditional P(X|Y) = P(X,Y) / P(Y)

**Chain rule:** P(A,B,C) = P(A|B,C) * P(B|C) * P(C)

**Naive Bayes** uses conditional independence to reduce parameters: P(X1...Xn|Y) = product of P(Xi|Y)

In [ ]:
# Discrete joint distribution: Weather x Commute
joint = np.array([[0.42, 0.08],  # sunny: [on-time, late]
                  [0.12, 0.38]]) # rainy:  [on-time, late]

print('Joint Distribution P(Weather, Commute):')
print('           On-time   Late   P(Weather)')
for i, w in enumerate(['Sunny', 'Rainy']):
    print(f"{w:8}   {joint[i,0]:6.2f}  {joint[i,1]:6.2f}   {joint[i].sum():.2f}")

mc = joint.sum(axis=0)
print(f"P(C):    {mc[0]:6.2f}  {mc[1]:6.2f}   {mc.sum():.2f}")

p_late_rainy = joint[1,1] / joint[1].sum()
p_late_sunny = joint[0,1] / joint[0].sum()
print(f"\nP(late | rainy): {p_late_rainy:.4f}")
print(f"P(late | sunny): {p_late_sunny:.4f}")
print(f"Rain increases lateness risk by {p_late_rainy/p_late_sunny:.1f}x")

### 2.4 MLE vs MAP Estimation

**MLE:** argmax_theta P(data | theta) -- no prior, can overfit with small data

**MAP:** argmax_theta [P(data|theta) * P(theta)] -- includes prior beliefs

**Regularization connection:**
- MAP with Gaussian prior = L2 regularization (Ridge)
- MAP with Laplace prior = L1 regularization (Lasso)

**Key insight:** With more data, MLE and MAP converge -- the prior gets dominated.

In [ ]:
# MLE vs MAP for coin flips (Beta-Binomial conjugate model)
# Beta(alpha, beta) prior; posterior is Beta(alpha+heads, beta+tails)
# MAP estimate = (alpha + heads - 1) / (n + alpha + beta - 2)

alpha_prior, beta_prior = 2, 2  # weak prior toward fair coin

scenarios = [
    (3, 5, 'Small sample'),
    (30, 50, 'Medium sample'),
    (300, 500, 'Large sample'),
]

print('MLE vs MAP for coin flip (true p=0.6):')
print('Scenario        heads/n    MLE     MAP')
for label, heads, n in scenarios:
    mle = heads / n
    map_est = (heads + alpha_prior - 1) / (n + alpha_prior + beta_prior - 2)
    print(f"{label:15} {heads}/{n:<4}     {mle:.4f}  {map_est:.4f}")

print('\nMAP shrinks toward prior mean (0.5) more with small data.')
print('With large data, both converge to the true proportion.')

# L2 regularization as MAP
print('\nConnection to regularization:')
print('MAP with Gaussian prior on weights -> L2 (Ridge) regularization')
print('MAP with Laplace prior on weights  -> L1 (Lasso) regularization')

### 2.5 Common Distributions -- Properties and ML Uses

| Distribution | Mean | Variance | Key ML Use |
|---|---|---|---|
| Normal(mu, sigma^2) | mu | sigma^2 | Weight initialization, noise modeling |
| Binomial(n,p) | np | np(1-p) | Binary classification output |
| Poisson(lambda) | lambda | lambda | Count data, language models |
| Exponential(lambda) | 1/lambda | 1/lambda^2 | Survival analysis, wait times |
| Beta(a,b) | a/(a+b) | -- | Prior over probabilities |
| Dirichlet(alpha) | -- | -- | Prior over categorical distributions |

In [ ]:
iris = load_iris()
sepal_length = iris.data[:, 0]

# Fit Normal distribution
mu, sigma = stats.norm.fit(sepal_length)
print(f"Iris sepal length: mean={mu:.3f}, std={sigma:.3f}")

# Test normality (Shapiro-Wilk)
stat, p_val = stats.shapiro(sepal_length[:50])
print(f"Shapiro-Wilk p-value: {p_val:.4f}")
print(f"Normal? {'Yes' if p_val > 0.05 else 'No'}")

# Poisson: mean == variance
samples = np.random.poisson(lam=3.0, size=10000)
print(f"\nPoisson(3): mean={samples.mean():.3f}, var={samples.var():.3f}")
print('For Poisson: mean == variance (useful as a diagnostic)')

# Central Limit Theorem demo
# Sample means of Exponential data become Normal
exp_means = [np.mean(np.random.exponential(2, 30)) for _ in range(2000)]
_, p_clt = stats.shapiro(exp_means[:50])
print(f"\nCLT demo: sample means of Exp(2) data -- Shapiro p={p_clt:.4f}")
print(f"Normal by CLT? {'Yes (p>0.05)' if p_clt > 0.05 else 'No'}")

---
## SECTION 3: Statistics Deep Dive

*Hypothesis testing, confidence intervals, and effect sizes are tested when you're applying for roles that involve A/B testing or experimentation platforms.*

### 3.1 Confidence Intervals -- Bootstrap Simulation

**Definition:** A 95% CI means: if we repeated the experiment 100 times, ~95 of the resulting intervals would contain the true parameter.

**Common misconception:** It does NOT mean 'there is a 95% probability the true value lies in this interval' (the true value is fixed, not random).

**Bootstrap:** Non-parametric CI method -- resample with replacement from your data to estimate the sampling distribution.

In [ ]:
import numpy as np
from scipy import stats

np.random.seed(42)
# True population: Normal(mu=5, sigma=2)
population = np.random.normal(5, 2, 10000)
sample = np.random.choice(population, size=100)

# Analytical CI (t-distribution)
n = len(sample)
mean = sample.mean()
se = sample.std(ddof=1) / np.sqrt(n)
t_crit = stats.t.ppf(0.975, df=n-1)
ci_low = mean - t_crit * se
ci_high = mean + t_crit * se

print(f"Sample mean: {mean:.4f}")
print(f"Standard error: {se:.4f}")
print(f"95% CI (t-dist): ({ci_low:.4f}, {ci_high:.4f})")
print(f"Contains true mean 5.0: {ci_low <= 5.0 <= ci_high}")

# Bootstrap CI
n_bootstrap = 10000
boot_means = [np.random.choice(sample, size=n, replace=True).mean()
              for _ in range(n_bootstrap)]
boot_ci = np.percentile(boot_means, [2.5, 97.5])

print(f"\n95% Bootstrap CI: ({boot_ci[0]:.4f}, {boot_ci[1]:.4f})")
print(f"Contains true mean 5.0: {boot_ci[0] <= 5.0 <= boot_ci[1]}")

# Verify coverage: run 1000 experiments
n_experiments = 1000
coverage = 0
for _ in range(n_experiments):
    s = np.random.choice(population, size=30)
    m = s.mean()
    se_s = s.std(ddof=1) / np.sqrt(30)
    t = stats.t.ppf(0.975, df=29)
    if m - t*se_s <= 5.0 <= m + t*se_s:
        coverage += 1
print(f"\nCI coverage verification: {coverage/n_experiments*100:.1f}% (target: 95%)")

### 3.2 Hypothesis Testing Framework

**Steps:**
1. State H0 (null) and H1 (alternative)
2. Choose significance level alpha (typically 0.05)
3. Compute test statistic
4. Compute p-value = P(data at least as extreme | H0 is true)
5. Reject H0 if p-value < alpha

**p-value misconception:** It is NOT the probability that H0 is true. It is the probability of observing results this extreme IF H0 were true.

In [ ]:
np.random.seed(42)

# A/B test: does new button color increase click rate?
# Control: 200 users, 40 clicks
# Treatment: 200 users, 55 clicks

n_control, clicks_control = 200, 40
n_treat, clicks_treat = 200, 55

p_control = clicks_control / n_control
p_treat = clicks_treat / n_treat

print(f"Control CTR: {p_control:.3f} ({clicks_control}/{n_control})")
print(f"Treatment CTR: {p_treat:.3f} ({clicks_treat}/{n_treat})")
print(f"Lift: {(p_treat-p_control)/p_control*100:.1f}%")

# Two-proportion z-test
p_pool = (clicks_control + clicks_treat) / (n_control + n_treat)
se = np.sqrt(p_pool*(1-p_pool)*(1/n_control + 1/n_treat))
z_stat = (p_treat - p_control) / se
p_value = 2 * (1 - stats.norm.cdf(abs(z_stat)))  # two-tailed

print(f"\nTest statistic z: {z_stat:.4f}")
print(f"p-value: {p_value:.4f}")
alpha = 0.05
print(f"Reject H0 (p < {alpha})? {p_value < alpha}")

# scipy built-in
chi2, p_chi, dof, expected = stats.chi2_contingency(
    [[clicks_control, n_control-clicks_control],
     [clicks_treat, n_treat-clicks_treat]]
)
print(f"\nchi2 test p-value: {p_chi:.4f} (same conclusion)")

### 3.3 Type I vs Type II Errors

| Error | Definition | Business Impact |
|---|---|---|
| Type I (False Positive) | Reject H0 when H0 is true | Ship a feature that doesn't work |
| Type II (False Negative) | Fail to reject H0 when H1 is true | Miss a feature that does work |

**Statistical Power** = 1 - P(Type II Error) = probability of detecting a real effect

**Trade-off:** Lowering alpha reduces Type I errors but increases Type II errors. You need larger samples to reduce both simultaneously.

In [ ]:
# Visualize Type I and Type II errors
# H0: mu=0, H1: mu=d (true effect size d)

alpha = 0.05
n = 50
sigma = 1.0

# Null distribution
se = sigma / np.sqrt(n)
z_crit = stats.norm.ppf(1 - alpha/2)  # two-tailed

print('Type I and Type II Error Analysis')
print('H0: mu=0, H1: mu=d, alpha=0.05, n=50')
print(f"Critical z-value: {z_crit:.4f}")
print(f"Reject region: |z| > {z_crit:.4f}")

# Power for different effect sizes
print('\nEffect size d   Power     Type II Error')
for d in [0.1, 0.2, 0.3, 0.5, 0.8, 1.0]:
    # Shift under H1
    ncp = d / se  # non-centrality parameter
    # Power = P(reject H0 | H1 true)
    power = (1 - stats.norm.cdf(z_crit - ncp) +
             stats.norm.cdf(-z_crit - ncp))
    print(f"  d={d:.1f}           {power:.4f}    {1-power:.4f}")

# Sample size needed for 80% power
from scipy.optimize import brentq
def power_for_n(n_val, d=0.3):
    se_n = sigma / np.sqrt(n_val)
    ncp = d / se_n
    return (1 - stats.norm.cdf(z_crit - ncp) + stats.norm.cdf(-z_crit - ncp)) - 0.8

n_needed = brentq(power_for_n, 2, 10000)
print(f"\nSample size needed for 80% power at d=0.3: {int(np.ceil(n_needed))}")

### 3.4 Effect Size -- Why p-value Alone Is Not Enough

**Problem:** With large enough samples, even tiny meaningless differences become statistically significant.

**Cohen's d:** Standardized effect size = (mean1 - mean2) / pooled_std

**Cohen's conventions:**
- d = 0.2: small effect
- d = 0.5: medium effect
- d = 0.8: large effect

**Interview answer:** Always report effect size alongside p-value. In ML: a model with statistically significantly better accuracy by 0.001% probably isn't worth deploying.

In [ ]:
# Effect size demonstration
np.random.seed(42)

def cohens_d(group1, group2):
    n1, n2 = len(group1), len(group2)
    var1, var2 = group1.var(ddof=1), group2.var(ddof=1)
    pooled_std = np.sqrt(((n1-1)*var1 + (n2-1)*var2) / (n1+n2-2))
    return (group1.mean() - group2.mean()) / pooled_std

# Small effect, large sample -> significant but meaningless
n_large = 10000
group_a_large = np.random.normal(5.000, 2, n_large)
group_b_large = np.random.normal(5.001, 2, n_large)  # 0.001 difference

t_stat, p_val = stats.ttest_ind(group_a_large, group_b_large)
d = cohens_d(group_a_large, group_b_large)
print(f"Large sample (n={n_large}) -- tiny true difference of 0.001:")
print(f"  p-value: {p_val:.4f}  -> Significant!")
print(f"  Cohen d: {d:.4f}  -> Negligible effect")

# Large effect, small sample -> not significant but real
n_small = 15
group_a_small = np.random.normal(5, 2, n_small)
group_b_small = np.random.normal(7, 2, n_small)  # large difference

t_stat2, p_val2 = stats.ttest_ind(group_a_small, group_b_small)
d2 = cohens_d(group_a_small, group_b_small)
print(f"\nSmall sample (n={n_small}) -- large true difference of 2.0:")
print(f"  p-value: {p_val2:.4f}  -> May not be significant")
print(f"  Cohen d: {d2:.4f}  -> Large effect")

print('\nKey: Always report both p-value AND effect size.')
print('p-value tells you: is there an effect?')
print('Effect size tells you: is the effect big enough to care about?')

### 3.5 Multiple Testing Problem and Bonferroni Correction

**Problem:** If you run 20 independent tests at alpha=0.05, the probability of at least one false positive is 1 - (0.95)^20 = 64%!

**Bonferroni correction:** Use alpha/m as threshold for each of m tests. Controls the Family-Wise Error Rate (FWER).

**Benjamini-Hochberg (FDR):** Less conservative -- controls the expected proportion of false discoveries among rejected hypotheses. Better when many tests are expected to be significant.

In [ ]:
np.random.seed(42)

# Simulate 100 hypothesis tests
# 90 null effects, 10 real effects
n_tests = 100
n_real = 10
alpha = 0.05

# Generate p-values
null_pvals = np.random.uniform(0, 1, n_tests - n_real)
real_pvals = np.random.beta(0.5, 10, n_real)  # skewed toward 0
all_pvals = np.concatenate([null_pvals, real_pvals])
is_real = np.array([False]*(n_tests-n_real) + [True]*n_real)

# No correction
rejected_naive = all_pvals < alpha
fp_naive = np.sum(rejected_naive & ~is_real)
tp_naive = np.sum(rejected_naive & is_real)

# Bonferroni correction
alpha_bonf = alpha / n_tests
rejected_bonf = all_pvals < alpha_bonf
fp_bonf = np.sum(rejected_bonf & ~is_real)
tp_bonf = np.sum(rejected_bonf & is_real)

# Benjamini-Hochberg
sorted_idx = np.argsort(all_pvals)
sorted_pvals = all_pvals[sorted_idx]
thresholds = (np.arange(1, n_tests+1) / n_tests) * alpha
bh_cutoff_idx = np.where(sorted_pvals <= thresholds)[0]
bh_rejected = np.zeros(n_tests, dtype=bool)
if len(bh_cutoff_idx) > 0:
    bh_rejected[sorted_idx[:bh_cutoff_idx[-1]+1]] = True
fp_bh = np.sum(bh_rejected & ~is_real)
tp_bh = np.sum(bh_rejected & is_real)

print('Multiple Testing Correction Results:')
print('Method        Rejected  True Pos  False Pos')
print(f"No correction   {rejected_naive.sum():4d}      {tp_naive:4d}       {fp_naive:4d}")
print(f"Bonferroni      {rejected_bonf.sum():4d}      {tp_bonf:4d}       {fp_bonf:4d}")
print(f"Benj-Hochberg   {bh_rejected.sum():4d}      {tp_bh:4d}       {fp_bh:4d}")

print(f"\nExpected false positives without correction:")
print(f"  1 - (0.95)^100 = {(1-0.95**100)*100:.1f}% chance of >= 1 FP")
print(f"  E[FP] = 0.05 * 90 null tests = {0.05*90:.1f}")

---
## SECTION 4: Information Theory

*Information theory concepts underpin loss functions, VAEs, feature selection, and generative models. Expect these in senior/research ML interviews.*

### 4.1 Entropy and Cross-Entropy

**Entropy:** H(P) = -sum_x P(x) * log P(x)
- Measures uncertainty/information content in a distribution
- Maximum for uniform distribution, zero for deterministic

**Cross-Entropy:** H(P, Q) = -sum_x P(x) * log Q(x)
- Measures how well Q approximates true distribution P
- Used as loss function in classification: H(y_true, y_pred)

**Note:** H(P,Q) = H(P) + KL(P||Q). Minimizing cross-entropy is equivalent to minimizing KL divergence when the true distribution P is fixed.

In [ ]:
def entropy(p):
    p = np.array(p, dtype=float)
    p = p[p > 0]
    return -np.sum(p * np.log2(p))

def cross_entropy(p_true, q_pred):
    p = np.array(p_true, dtype=float)
    q = np.array(q_pred, dtype=float)
    q = np.clip(q, 1e-10, 1)
    return -np.sum(p * np.log2(q))

# Entropy examples
uniform_4 = [0.25, 0.25, 0.25, 0.25]
certain = [1.0, 0.0, 0.0, 0.0]
skewed = [0.7, 0.1, 0.1, 0.1]

print('Entropy (bits):')
print(f"Uniform [.25,.25,.25,.25]: {entropy(uniform_4):.4f} bits (max)")
print(f"Certain [1.0, 0, 0, 0]:   {entropy(certain):.4f} bits (min)")
print(f"Skewed  [.7,.1,.1,.1]:     {entropy(skewed):.4f} bits")

# Cross-entropy as classification loss
y_true = [0, 0, 1, 0]  # one-hot: class 3
y_pred_good = [0.02, 0.02, 0.94, 0.02]
y_pred_bad  = [0.25, 0.25, 0.25, 0.25]

print('\nCross-Entropy (classification loss):')
print(f"Good prediction:   {cross_entropy(y_true, y_pred_good):.4f}")
print(f"Random prediction: {cross_entropy(y_true, y_pred_bad):.4f}")

# Binary cross-entropy
print('\nBinary cross-entropy (logistic loss):')
for y, yhat in [(1, 0.9), (1, 0.5), (1, 0.1)]:
    bce = -(y * np.log(yhat) + (1-y) * np.log(1-yhat))
    print(f"  y=1, pred={yhat}: BCE={bce:.4f}")

### 4.2 KL Divergence

**Formula:** KL(P||Q) = sum_x P(x) * log(P(x)/Q(x))

**Properties:**
- Always >= 0 (Gibbs inequality)
- KL(P||Q) != KL(Q||P) -- NOT symmetric
- KL = 0 iff P = Q

**Use in VAEs:** The ELBO loss = reconstruction loss + KL(q(z|x) || p(z))
The KL term regularizes the latent space toward the prior N(0,I).

**Forward vs reverse KL:**
- KL(P||Q): 'forward' -- forces Q to cover all modes of P (mean-seeking)
- KL(Q||P): 'reverse' -- forces Q to concentrate on modes of P (mode-seeking)

In [ ]:
def kl_divergence(p, q):
    p = np.array(p, dtype=float)
    q = np.array(q, dtype=float)
    q = np.clip(q, 1e-10, 1)
    mask = p > 0
    return np.sum(p[mask] * np.log2(p[mask] / q[mask]))

p = [0.4, 0.35, 0.15, 0.1]
q = [0.25, 0.25, 0.25, 0.25]  # uniform
r = [0.3, 0.4, 0.2, 0.1]      # close to p

print(f"KL(P||Q_uniform):   {kl_divergence(p, q):.4f}")
print(f"KL(Q_uniform||P):   {kl_divergence(q, p):.4f}")
print(f"KL(P||R_close):     {kl_divergence(p, r):.4f}")
print(f"KL(P||P):           {kl_divergence(p, p):.4f} (always 0)")

# KL for continuous Gaussians: closed form
# KL(N(mu1,s1^2) || N(mu2,s2^2))
def kl_gaussian(mu1, s1, mu2, s2):
    return (np.log(s2/s1) + (s1**2 + (mu1-mu2)**2)/(2*s2**2) - 0.5)

print('\nKL between Gaussian distributions:')
print(f"KL(N(0,1) || N(0,1)):     {kl_gaussian(0,1,0,1):.4f}")
print(f"KL(N(1,1) || N(0,1)):     {kl_gaussian(1,1,0,1):.4f}")
print(f"KL(N(0,2) || N(0,1)):     {kl_gaussian(0,2,0,1):.4f}")

# VAE regularization: KL(N(mu,sigma^2) || N(0,1))
# Closed form: 0.5 * (sigma^2 + mu^2 - 1 - log(sigma^2))
mu_enc, sigma_enc = 0.5, 0.8
kl_vae = 0.5 * (sigma_enc**2 + mu_enc**2 - 1 - np.log(sigma_enc**2))
print(f"\nVAE KL term for encoder N({mu_enc},{sigma_enc}^2): {kl_vae:.4f}")
print('This pushes encoded distribution toward standard Normal N(0,1)')

### 4.3 Mutual Information for Feature Selection

**Formula:** I(X;Y) = H(X) - H(X|Y) = H(Y) - H(Y|X)

Measures how much knowing X reduces uncertainty about Y.

**vs Correlation:** Mutual information captures non-linear relationships; Pearson correlation only captures linear.

**Use cases:**
- Feature selection: select features with high MI toward target
- Representation learning: maximize MI between input and learned representation
- InfoGAN: maximize MI between latent code and generated images

In [ ]:
from sklearn.feature_selection import mutual_info_classif
from sklearn.datasets import load_iris

iris = load_iris()
X, y = iris.data, iris.target
feature_names = iris.feature_names

# Mutual information between each feature and target
mi_scores = mutual_info_classif(X, y, random_state=42)

print('Mutual Information between features and Iris target:')
ranked = np.argsort(mi_scores)[::-1]
for i in ranked:
    print(f"  {feature_names[i]:28}: {mi_scores[i]:.4f}")

# Compare: Pearson correlation (linear) vs MI (nonlinear)
print('\nPearson correlation vs Mutual Information:')
for i, fname in enumerate(feature_names):
    r, _ = stats.pearsonr(X[:, i], y)
    print(f"  {fname[:20]:22} | r={r:+.4f} | MI={mi_scores[i]:.4f}")

# Synthetic nonlinear example
x_nl = np.linspace(-3, 3, 500)
y_lin = 2*x_nl + np.random.normal(0, 0.1, 500)
y_nl  = x_nl**2 + np.random.normal(0, 0.1, 500)

r_lin, _ = stats.pearsonr(x_nl, y_lin)
r_nl, _  = stats.pearsonr(x_nl, y_nl)

print('\nNonlinear relationship detection:')
print(f"  y = 2x:    Pearson r = {r_lin:.4f}  (correctly detected)")
print(f"  y = x^2:   Pearson r = {r_nl:.4f}  (missed! near zero)")
print('  Mutual information would detect both relationships')

---
## SECTION 5: 20 Real ML Interview Questions with Answers

*Questions compiled from reported FAANG interviews, Glassdoor, and Chip Huyen's book.*

Format: Question -> Plain English Answer -> Code Proof

### Q1: What is the curse of dimensionality?

**Question (Google, Meta):** Explain the curse of dimensionality and its implications for ML models.

**Answer:** In high dimensions, data becomes sparse -- the volume of the space grows exponentially, so your fixed number of data points covers a vanishingly small fraction. Consequences:
- Distance metrics become meaningless (all points are equidistant)
- More data needed to maintain the same density
- Overfitting risk increases
- KNN and kernel methods degrade

In [ ]:
import numpy as np

# Distance concentration in high dimensions
# All pairwise distances converge to the same value
np.random.seed(42)

print('Relative std of pairwise distances (lower = more concentrated):')
print('Dimension   Mean Dist   Std Dev   Std/Mean')
for d in [2, 5, 10, 50, 100, 500, 1000]:
    points = np.random.randn(200, d)
    # Compute pairwise distances (subsample)
    dists = []
    for i in range(100):
        j = (i+1) % 200
        dists.append(np.linalg.norm(points[i] - points[j]))
    dists = np.array(dists)
    rel_std = dists.std() / dists.mean()
    print(f"  d={d:5d}       {dists.mean():8.2f}    {dists.std():6.3f}     {rel_std:.4f}")

print('\nAs d increases, relative std -> 0: all distances look the same!')
print('This breaks nearest-neighbor algorithms in high dimensions.')

### Q2: When would you use L1 vs L2 regularization?

**Question (Amazon, Apple):**

**Answer:**
- **L2 (Ridge):** Adds sum(w^2) to loss. Shrinks all weights toward zero but rarely to exactly zero. Works well when most features are relevant.
- **L1 (Lasso):** Adds sum(|w|) to loss. Produces sparse solutions (many weights become exactly zero). Works well for feature selection.
- **Elastic Net:** Combination of L1 and L2. Best of both worlds.

**Math reason L1 produces sparsity:** The L1 constraint ball has corners at the axes. Loss function contours tend to hit these corners first.

In [ ]:
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.datasets import make_regression

np.random.seed(42)
X, y, true_coef = make_regression(n_samples=100, n_features=20,
                                   n_informative=5, noise=10,
                                   coef=True, random_state=42)

# Fit models
ridge = Ridge(alpha=1.0).fit(X, y)
lasso = Lasso(alpha=1.0, max_iter=10000).fit(X, y)
enet  = ElasticNet(alpha=1.0, l1_ratio=0.5, max_iter=10000).fit(X, y)

print('Number of zero coefficients (sparsity):')
print(f"True model:   {np.sum(true_coef == 0)} zeros out of 20 features")
print(f"Ridge:        {np.sum(np.abs(ridge.coef_) < 0.01)} near-zero coefs")
print(f"Lasso:        {np.sum(lasso.coef_ == 0.0)} exact zeros")
print(f"Elastic Net:  {np.sum(enet.coef_ == 0.0)} exact zeros")

print('\nCoefficient magnitudes (first 10 features):')
print('Feature  True       Ridge      Lasso      ElasticNet')
for i in range(10):
    r = ridge.coef_[i]; l = lasso.coef_[i]; e = enet.coef_[i]; tc = true_coef[i]
    print(f"{i:5d}    {tc:8.3f}   {r:8.3f}   {l:8.3f}   {e:8.3f}")

### Q3: Why is the sigmoid function problematic for deep networks?

**Question (Google DeepMind, OpenAI):**

**Answer:** The **vanishing gradient problem**. Sigmoid saturates at 0 and 1 where the gradient is ~0. During backpropagation, gradients are multiplied at each layer. With many layers, gradients become exponentially small, making early layers learn very slowly.

**Solution:** ReLU (gradient = 1 in positive region) avoids saturation. But ReLU has 'dying ReLU' problem. Leaky ReLU, GELU, and Swish are modern alternatives.

In [ ]:
# Sigmoid gradient analysis
x = np.linspace(-6, 6, 1000)

sigmoid = 1 / (1 + np.exp(-x))
sigmoid_grad = sigmoid * (1 - sigmoid)  # derivative

relu = np.maximum(0, x)
relu_grad = (x > 0).astype(float)

print('Sigmoid gradient analysis:')
print(f"Max sigmoid gradient: {sigmoid_grad.max():.4f} (at x=0)")
print(f"Sigmoid gradient at x=4: {sigmoid_grad[np.argmin(np.abs(x-4))]:.6f}")
print(f"Sigmoid gradient at x=6: {sigmoid_grad[np.argmin(np.abs(x-6))]:.8f}")

# Vanishing gradient simulation: 10-layer sigmoid network
x_in = 3.0  # activated in saturating region
grad = 1.0
print('\nGradient magnitude through 10 sigmoid layers:')
print('(starting input x=3.0, all layers saturated)')
for layer in range(1, 11):
    s = 1 / (1 + np.exp(-x_in))
    local_grad = s * (1 - s)
    grad *= local_grad
    print(f"  After layer {layer:2d}: gradient = {grad:.2e}")

print('\nWith ReLU (positive region): gradient stays 1.0 through all layers')

### Q4: What is the bias-variance tradeoff?

**Question (Meta, Netflix -- virtually universal):**

**Answer:** Total expected error = Bias^2 + Variance + Irreducible Noise

- **Bias:** Error from wrong assumptions (underfitting)
- **Variance:** Error from sensitivity to training data fluctuations (overfitting)

| Model | Bias | Variance |
|---|---|---|
| Linear regression | High | Low |
| Deep neural net (unregularized) | Low | High |
| Random Forest (many trees) | Low | Lower than single tree |

**Modern insight:** Very large models (foundation models) can have both low bias AND low variance -- the 'double descent' phenomenon.

In [ ]:
np.random.seed(42)

# Bias-variance decomposition via simulation
def true_f(x):
    return np.sin(x)

def estimate_bias_variance(degree, n_samples=30, n_experiments=200):
    x_test = np.array([1.5])
    predictions = []
    for _ in range(n_experiments):
        x_train = np.random.uniform(-3, 3, n_samples)
        y_train = true_f(x_train) + np.random.normal(0, 0.3, n_samples)
        coefs = np.polyfit(x_train, y_train, degree)
        pred = np.polyval(coefs, x_test[0])
        predictions.append(pred)
    preds = np.array(predictions)
    bias2 = (preds.mean() - true_f(x_test[0]))**2
    variance = preds.var()
    return bias2, variance

true_val = true_f(1.5)
print('Bias-Variance Decomposition (predicting sin(1.5)):')
print(f"True value: {true_val:.4f}")
print('Degree  Bias^2     Variance   Total')
for d in [1, 3, 5, 10, 15]:
    b2, v = estimate_bias_variance(d)
    print(f"  {d:2d}    {b2:.6f}   {v:.6f}   {b2+v:.6f}")

print('\nLow degree = high bias (underfitting)')
print('High degree = high variance (overfitting)')
print('Optimal degree = best bias-variance tradeoff')

### Q5: Derive the formula for linear regression (normal equation)

**Question (Google, Microsoft):**

**Answer:** Minimize MSE: L = ||y - Xw||^2

Take derivative and set to zero:
dL/dw = -2 X^T(y - Xw) = 0
=> X^T X w = X^T y
=> w* = (X^T X)^-1 X^T y

**Complexity:** O(n * d^2 + d^3) -- quadratic in features d, linear in samples n. Gradient descent is preferred when d is large (e.g., d > 10,000).

In [ ]:
np.random.seed(42)

# Generate regression data
n, d = 100, 3
X_raw = np.random.randn(n, d)
X = np.column_stack([np.ones(n), X_raw])  # add bias column
true_w = np.array([2.0, 1.5, -0.5, 0.8])
y = X @ true_w + np.random.normal(0, 0.5, n)

# Normal equation
w_normal = np.linalg.inv(X.T @ X) @ X.T @ y

# Gradient descent
w_gd = np.zeros(d+1)
lr = 0.01
for _ in range(2000):
    residuals = y - X @ w_gd
    grad = -2 * X.T @ residuals / n
    w_gd -= lr * grad

print('True weights:    ', true_w.round(4))
print('Normal equation: ', w_normal.round(4))
print('Gradient descent:', w_gd.round(4))

mse_normal = np.mean((y - X @ w_normal)**2)
mse_gd     = np.mean((y - X @ w_gd)**2)
print(f"\nMSE (normal eq): {mse_normal:.6f}")
print(f"MSE (grad desc): {mse_gd:.6f}")

# sklearn verification
from sklearn.linear_model import LinearRegression
lr_model = LinearRegression(fit_intercept=False).fit(X, y)
print('sklearn coefs:   ', lr_model.coef_.round(4))

### Q6: What does AUC-ROC measure? When is accuracy a bad metric?

**Question (Meta, Airbnb):**

**Answer:**
- **AUC-ROC** = probability that the model ranks a random positive example higher than a random negative example. Threshold-independent.
- **Accuracy is misleading** when classes are imbalanced. A model predicting 'not fraud' always achieves 99.9% accuracy on fraud data but is completely useless.

**Better metrics for imbalanced data:** Precision, Recall, F1, AUC-ROC, AUC-PR.
**AUC-PR** is better than AUC-ROC when positive class is rare.

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.metrics import confusion_matrix, classification_report

np.random.seed(42)

# Imbalanced dataset: 1% positive (fraud)
n_total = 10000
n_pos = 100
n_neg = n_total - n_pos

y_true = np.array([1]*n_pos + [0]*n_neg)

# Model A: always predicts 0 (majority class)
y_pred_A = np.zeros(n_total, dtype=int)
# Model B: actual classifier with some skill
scores_B = np.concatenate([
    np.random.beta(5, 2, n_pos),   # positives get high scores
    np.random.beta(2, 5, n_neg)    # negatives get low scores
])
y_pred_B = (scores_B > 0.5).astype(int)

acc_A = np.mean(y_pred_A == y_true)
acc_B = np.mean(y_pred_B == y_true)

print(f"Imbalanced dataset: {n_pos} positives, {n_neg} negatives")
print(f"\nModel A (always predict 0): accuracy={acc_A:.4f}")
print(f"Model B (actual classifier): accuracy={acc_B:.4f}")

# AUC-ROC (need probability scores for A, set all to 0)
auc_A = 0.5  # random classifier
auc_B = roc_auc_score(y_true, scores_B)

print(f"\nModel A AUC-ROC: {auc_A:.4f} (random)")
print(f"Model B AUC-ROC: {auc_B:.4f}")

# Precision/Recall
from sklearn.metrics import precision_score, recall_score, f1_score
prec = precision_score(y_true, y_pred_B)
rec  = recall_score(y_true, y_pred_B)
f1   = f1_score(y_true, y_pred_B)
print(f"\nModel B -- Precision: {prec:.4f}, Recall: {rec:.4f}, F1: {f1:.4f}")
print('\nModel A has high accuracy but catches ZERO fraud cases!')
print('AUC and F1 correctly reveal Model B is superior.')

### Q7: What is the difference between bagging and boosting?

**Question (Amazon, LinkedIn):**

**Bagging (Bootstrap Aggregating):**
- Train models in parallel on bootstrap samples
- Reduces variance (good for high-variance models like deep trees)
- Example: Random Forest

**Boosting:**
- Train models sequentially, each focusing on errors of previous
- Reduces bias (good for weak learners)
- Example: AdaBoost, Gradient Boosting, XGBoost

**Interview tip:** Random Forest = bagging + feature subsampling. XGBoost = gradient boosting + regularization + efficient tree building.

In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score
from sklearn.datasets import load_iris

iris = load_iris()
X_iris, y_iris = iris.data, iris.target

models = {
    'Single Tree (depth=1)': DecisionTreeClassifier(max_depth=1, random_state=42),
    'Single Tree (deep)':    DecisionTreeClassifier(random_state=42),
    'Random Forest (bag)':   RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting':     GradientBoostingClassifier(n_estimators=100, random_state=42),
}

print('5-fold CV accuracy on Iris:')
print('Model                       Mean    Std')
for name, model in models.items():
    scores = cross_val_score(model, X_iris, y_iris, cv=5)
    print(f"{name:30} {scores.mean():.4f}  {scores.std():.4f}")

print('\nKey insight:')
print('- Single shallow tree: high bias (underfitting)')
print('- Single deep tree: high variance')
print('- Random Forest: reduces variance via bagging')
print('- Gradient Boosting: reduces bias via sequential correction')

### Q8: Explain gradient descent and its variants

**Question (Google, OpenAI):**

**Variants:**
- **Batch GD:** Use all data. Stable but slow and memory-intensive
- **SGD:** Use one sample. Fast but noisy
- **Mini-batch GD:** Use k samples. Best of both worlds (standard practice)
- **Momentum:** Accumulate velocity in gradient direction. Faster convergence
- **Adam:** Adaptive learning rates per parameter + momentum. Default in DL

**Adam formula:** w = w - lr * m_hat / (sqrt(v_hat) + epsilon)
where m_hat and v_hat are bias-corrected first and second moment estimates.

In [ ]:
np.random.seed(42)

# Compare optimizers on a simple 2D loss surface
def rosenbrock(w):
    # Modified Rosenbrock -- harder optimization problem
    x, y = w[0], w[1]
    return (1 - x)**2 + 100*(y - x**2)**2

def rosenbrock_grad(w):
    x, y = w[0], w[1]
    gx = -2*(1-x) - 400*x*(y - x**2)
    gy = 200*(y - x**2)
    return np.array([gx, gy])

def sgd(w0, lr, n_steps):
    w = w0.copy()
    losses = [rosenbrock(w)]
    for _ in range(n_steps):
        g = rosenbrock_grad(w)
        w -= lr * g
        losses.append(rosenbrock(w))
    return w, losses

def adam(w0, lr, n_steps, b1=0.9, b2=0.999, eps=1e-8):
    w = w0.copy()
    m = np.zeros_like(w)
    v = np.zeros_like(w)
    losses = [rosenbrock(w)]
    for t in range(1, n_steps+1):
        g = rosenbrock_grad(w)
        m = b1*m + (1-b1)*g
        v = b2*v + (1-b2)*g**2
        m_hat = m / (1 - b1**t)
        v_hat = v / (1 - b2**t)
        w -= lr * m_hat / (np.sqrt(v_hat) + eps)
        losses.append(rosenbrock(w))
    return w, losses

w0 = np.array([-0.5, 0.5])
n_steps = 5000

w_sgd, losses_sgd  = sgd(w0, lr=0.001, n_steps=n_steps)
w_adam, losses_adam = adam(w0, lr=0.01, n_steps=n_steps)

print('Optimizer comparison on Rosenbrock function:')
print(f"True minimum: (1.0, 1.0), f=0.0")
print(f"\nSGD (lr=0.001):  final w={w_sgd.round(4)}, loss={losses_sgd[-1]:.4f}")
print(f"Adam (lr=0.01):  final w={w_adam.round(4)}, loss={losses_adam[-1]:.6f}")

# Loss at milestone steps
print('\nLoss at milestones:')
print('Steps    SGD        Adam')
for step in [100, 500, 1000, 5000]:
    print(f"{step:5d}    {losses_sgd[step]:.4f}     {losses_adam[step]:.6f}")

### Q9: What is regularization and why does it work?

**Question (Meta, Microsoft):**

**Answer:** Regularization adds a penalty to the loss function to constrain model complexity, reducing overfitting.

**Why it works (information theory view):** A model with fewer effective parameters requires less data to fit well. Regularization reduces the effective complexity.

**Other forms of regularization:**
- Dropout: randomly zeroes activations during training
- Batch Normalization: reduces internal covariate shift, slight regularization effect
- Data augmentation: artificially increases training set diversity
- Early stopping: stop before overfitting

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Ridge, Lasso
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

np.random.seed(42)

# Generate data from true quadratic
x = np.linspace(-3, 3, 100)
y_true = 0.5 * x**2 - x + 2
y = y_true + np.random.normal(0, 0.5, 100)
X_1d = x.reshape(-1, 1)

X_train, X_test, y_train, y_test = train_test_split(X_1d, y, test_size=0.3, random_state=42)

results = {}
for degree in [2, 10, 20]:
    for reg_name, reg_model in [
        ('None', Ridge(alpha=0.0001)),
        ('L2(1)', Ridge(alpha=1.0)),
        ('L2(100)', Ridge(alpha=100.0)),
    ]:
        pipe = Pipeline([
            ('poly', PolynomialFeatures(degree=degree)),
            ('model', reg_model)
        ])
        pipe.fit(X_train, y_train)
        train_mse = np.mean((pipe.predict(X_train) - y_train)**2)
        test_mse  = np.mean((pipe.predict(X_test) - y_test)**2)
        results[(degree, reg_name)] = (train_mse, test_mse)

print('Regularization Effect on Polynomial Regression:')
print('Degree  Regularization  Train MSE   Test MSE')
for (d, r), (tr, te) in sorted(results.items()):
    print(f"  {d:2d}    {r:12}    {tr:8.4f}    {te:8.4f}")

### Q10: Explain how PCA works, step by step

**Question (Google, Meta, academia):**

**Steps:**
1. Center the data (subtract mean)
2. Compute covariance matrix C = X^T X / (n-1)
3. Compute eigendecomposition of C
4. Sort eigenvectors by eigenvalue (descending)
5. Project data onto top-k eigenvectors

**Equivalent via SVD:** X = U * Sigma * V^T. Principal components = columns of V. Scores = U * Sigma. More numerically stable than eigendecomposition of X^T X.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler

iris = load_iris()
X_iris, y_iris = iris.data, iris.target

# Manual PCA
X_c = X_iris - X_iris.mean(axis=0)
C = X_c.T @ X_c / (len(X_c) - 1)
evals, evecs = np.linalg.eigh(C)
idx = np.argsort(evals)[::-1]
evals, evecs = evals[idx], evecs[:, idx]

X_manual = X_c @ evecs[:, :2]  # project to 2D

# sklearn PCA
pca = PCA(n_components=2)
X_sklearn = pca.fit_transform(X_iris)

print(f"Variance explained:")
for i, (ev, pv) in enumerate(zip(
    evals / evals.sum(),
    pca.explained_variance_ratio_
), 1):
    print(f"  PC{i}: manual={ev:.4f}, sklearn={pv:.4f}")

total_ev = np.sum(pca.explained_variance_ratio_)
print(f"\nTotal variance in 2D: {total_ev*100:.1f}%")

# Check equivalence (may differ by sign flip)
corr = np.corrcoef(np.abs(X_manual[:, 0]), np.abs(X_sklearn[:, 0]))[0,1]
print(f"Manual vs sklearn PC1 correlation: {corr:.6f}")

### Q11: What is the kernel trick in SVMs?

**Question (Google, research roles):**

**Answer:** Instead of explicitly mapping data to a high-dimensional space (which could be infinite-dimensional), the kernel trick computes the dot product in that space implicitly: K(x, x') = phi(x) . phi(x')

This allows SVMs to find nonlinear decision boundaries while only computing kernel evaluations (O(n) operations per prediction), not explicit feature maps.

**Common kernels:**
- Linear: K(x,x') = x.x'
- RBF/Gaussian: K(x,x') = exp(-gamma||x-x'||^2) -- corresponds to infinite-dimensional feature space
- Polynomial: K(x,x') = (x.x' + c)^d

In [ ]:
from sklearn.svm import SVC
from sklearn.datasets import make_moons
from sklearn.model_selection import cross_val_score

np.random.seed(42)
X_moons, y_moons = make_moons(n_samples=500, noise=0.2, random_state=42)

kernels = {
    'linear': SVC(kernel='linear', random_state=42),
    'rbf (gamma=auto)': SVC(kernel='rbf', random_state=42),
    'poly (d=3)': SVC(kernel='poly', degree=3, random_state=42),
}

print('SVM kernel comparison on moons dataset (5-fold CV):')
print('Kernel              Mean Acc   Std')
for name, clf in kernels.items():
    scores = cross_val_score(clf, X_moons, y_moons, cv=5)
    print(f"{name:20} {scores.mean():.4f}     {scores.std():.4f}")

print('\nLinear SVM cannot fit the nonlinear moons boundary.')
print('RBF kernel implicitly maps to infinite dimensions -> separable!')

### Q12: How does a Random Forest reduce variance compared to a single tree?

**Answer:** Through two sources of randomness:
1. **Bootstrap sampling:** Each tree trains on a different random sample with replacement
2. **Feature subsampling:** Each split considers only sqrt(d) random features

If trees were identical, averaging wouldn't help. The correlation between trees limits variance reduction: Var(average) = rho * sigma^2 + (1-rho)/n * sigma^2

where rho is correlation between trees. Feature subsampling reduces rho.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import make_classification

np.random.seed(42)
X_cls, y_cls = make_classification(n_samples=1000, n_features=20,
                                    n_informative=10, random_state=42)

# Variance of single trees vs ensemble
n_runs = 30
tree_accs = []
forest_accs = []

for seed in range(n_runs):
    X_tr, X_te = X_cls[:800], X_cls[800:]
    y_tr, y_te = y_cls[:800], y_cls[800:]

    # Single tree
    dt = DecisionTreeClassifier(random_state=seed).fit(X_tr, y_tr)
    tree_accs.append(dt.score(X_te, y_te))

    # Random forest
    rf = RandomForestClassifier(n_estimators=100, random_state=seed).fit(X_tr, y_tr)
    forest_accs.append(rf.score(X_te, y_te))

tree_accs = np.array(tree_accs)
forest_accs = np.array(forest_accs)

print('Accuracy across 30 runs (different random seeds):')
print(f"Single Tree:   mean={tree_accs.mean():.4f}, std={tree_accs.std():.4f}")
print(f"Random Forest: mean={forest_accs.mean():.4f}, std={forest_accs.std():.4f}")
print(f"\nVariance reduction: {tree_accs.var()/forest_accs.var():.1f}x")
print('Random Forest has similar mean but much lower variance.')

### Q13: What is the difference between precision and recall? When to optimize each?

**Answer:**
- **Precision** = TP / (TP + FP): Of all predicted positives, how many are real?
- **Recall** = TP / (TP + FN): Of all actual positives, how many did we find?

**When to optimize:**
- High precision: Spam filter (false positives annoy users by blocking good mail)
- High recall: Cancer screening (false negatives mean missing cancer)
- F1 = harmonic mean of precision and recall (balanced)

**PR curve:** Shows precision-recall tradeoff at different thresholds. AUC-PR is better than AUC-ROC for imbalanced problems.

In [ ]:
from sklearn.metrics import precision_recall_curve, roc_curve
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification

np.random.seed(42)
X_bin, y_bin = make_classification(n_samples=1000, weights=[0.95, 0.05],
                                    random_state=42)

from sklearn.model_selection import train_test_split
Xtr, Xte, ytr, yte = train_test_split(X_bin, y_bin, test_size=0.3, random_state=42)

model = LogisticRegression(random_state=42, max_iter=1000).fit(Xtr, ytr)
probs = model.predict_proba(Xte)[:, 1]

precisions, recalls, thresholds = precision_recall_curve(yte, probs)

print('Precision-Recall at different thresholds:')
print('Threshold   Precision   Recall     F1')
for t in [0.1, 0.3, 0.5, 0.7, 0.9]:
    preds = (probs >= t).astype(int)
    tp = np.sum((preds==1) & (yte==1))
    fp = np.sum((preds==1) & (yte==0))
    fn = np.sum((preds==0) & (yte==1))
    prec = tp/(tp+fp) if (tp+fp)>0 else 0
    rec  = tp/(tp+fn) if (tp+fn)>0 else 0
    f1   = 2*prec*rec/(prec+rec) if (prec+rec)>0 else 0
    print(f"  {t:.1f}         {prec:.4f}      {rec:.4f}     {f1:.4f}")

print('\nLowering threshold -> higher recall, lower precision (more FPs)')
print('Raising threshold  -> higher precision, lower recall (more FNs)')

### Q14: Explain backpropagation

**Question (Google, Meta, OpenAI -- nearly universal):**

**Answer:** Backpropagation is the algorithm for computing gradients of the loss with respect to all parameters in a neural network using the chain rule.

**Forward pass:** Compute activations layer by layer, cache intermediate values

**Backward pass:** Apply chain rule from output to input:
dL/dW_i = dL/da_{i+1} * da_{i+1}/dz_{i+1} * dz_{i+1}/dW_i

**Key insight:** Reusing cached forward-pass values makes this O(n) in network size rather than O(n^2).

In [ ]:
# Manual backpropagation on a simple 1-hidden-layer network
np.random.seed(42)

# Data
X_bp = np.random.randn(100, 2)
y_bp = (X_bp[:, 0] + X_bp[:, 1] > 0).astype(float).reshape(-1, 1)

# Initialize
W1 = np.random.randn(2, 4) * 0.1
b1 = np.zeros((1, 4))
W2 = np.random.randn(4, 1) * 0.1
b2 = np.zeros((1, 1))

def sigmoid_fn(z):
    return 1 / (1 + np.exp(-z))

def forward(X, W1, b1, W2, b2):
    z1 = X @ W1 + b1
    a1 = np.tanh(z1)
    z2 = a1 @ W2 + b2
    a2 = sigmoid_fn(z2)
    return z1, a1, z2, a2

lr = 0.1
losses = []
for epoch in range(200):
    # Forward
    z1, a1, z2, a2 = forward(X_bp, W1, b1, W2, b2)
    loss = -np.mean(y_bp * np.log(a2+1e-10) + (1-y_bp) * np.log(1-a2+1e-10))
    losses.append(loss)

    # Backward (chain rule)
    n = len(X_bp)
    dz2 = a2 - y_bp                          # dL/dz2
    dW2 = a1.T @ dz2 / n                    # dL/dW2
    db2 = dz2.mean(axis=0, keepdims=True)
    da1 = dz2 @ W2.T                         # dL/da1
    dz1 = da1 * (1 - a1**2)                 # dL/dz1 (tanh deriv)
    dW1 = X_bp.T @ dz1 / n
    db1 = dz1.mean(axis=0, keepdims=True)

    # Update
    W1 -= lr * dW1
    b1 -= lr * db1
    W2 -= lr * dW2
    b2 -= lr * db2

_, _, _, final_pred = forward(X_bp, W1, b1, W2, b2)
acc = np.mean((final_pred.flatten() > 0.5) == y_bp.flatten())
print(f"Manual backprop network (200 epochs):")
print(f"Initial loss: {losses[0]:.4f}")
print(f"Final loss:   {losses[-1]:.4f}")
print(f"Accuracy:     {acc:.4f}")

### Q15: What is the softmax function and why is it used?

**Answer:** Softmax(z_i) = exp(z_i) / sum_j exp(z_j)

Converts raw logits into a probability distribution (all values positive, sum to 1).

**Properties:**
- Amplifies the largest value (soft argmax)
- Differentiable everywhere
- Temperature scaling: Softmax(z/T) -- higher T = softer distribution

**Numerical stability:** Always subtract max(z) before exponentiating: exp(z-max(z)) / sum(exp(z-max(z))). Same result, avoids overflow.

In [ ]:
def softmax(z, temperature=1.0):
    z_scaled = z / temperature
    z_stable = z_scaled - np.max(z_scaled)  # numerical stability
    exp_z = np.exp(z_stable)
    return exp_z / exp_z.sum()

logits = np.array([2.0, 1.0, 0.5, -1.0])

print('Softmax with different temperatures:')
print('Temperature   Class probabilities       Entropy')
for T in [0.1, 0.5, 1.0, 2.0, 10.0]:
    probs = softmax(logits, T)
    H = -np.sum(probs * np.log(probs + 1e-10))
    print(f"  T={T:4.1f}       [{" ".join([f"{p:.3f}" for p in probs])}]   {H:.4f}")

print('\nLow T: picks argmax (confident). High T: approaches uniform (uncertain).')
print('Used in: distillation (T>1), sampling (T varies), inference (T=1)')

# Numerical stability demonstration
large_logits = np.array([1000.0, 1001.0, 999.0])
print('\nNumerical stability with large logits:')
try:
    exp_direct = np.exp(large_logits)
    naive = exp_direct / exp_direct.sum()
    print(f'Naive softmax: {naive}')
except Exception as e:
    print(f'Naive fails: {e}')
stable = softmax(large_logits)
print(f"Stable softmax: {stable.round(4)}")

### Q16: What is a confusion matrix and how do you derive metrics from it?

**Answer:**
```
              Predicted Positive  Predicted Negative
Actual Positive    TP                  FN
Actual Negative    FP                  TN
```

**Derived metrics:**
- Accuracy = (TP + TN) / total
- Precision = TP / (TP + FP)
- Recall (Sensitivity) = TP / (TP + FN)
- Specificity = TN / (TN + FP)
- F1 = 2 * Precision * Recall / (Precision + Recall)
- MCC = (TP*TN - FP*FN) / sqrt((TP+FP)(TP+FN)(TN+FP)(TN+FN))

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.metrics import matthews_corrcoef

# Simulate predictions
np.random.seed(42)
y_true_ex = np.array([1,1,1,1,1,1,1,1,1,1,0,0,0,0,0,0,0,0,0,0])
y_pred_ex = np.array([1,1,1,1,1,0,0,1,1,0,0,0,0,0,1,1,0,0,0,0])

cm = confusion_matrix(y_true_ex, y_pred_ex)
TP, FN = cm[1,1], cm[1,0]
FP, TN = cm[0,1], cm[0,0]

print('Confusion Matrix:')
print(f'  TP={TP}, FN={FN}')
print(f'  FP={FP}, TN={TN}')

accuracy  = (TP+TN) / (TP+TN+FP+FN)
precision = TP / (TP+FP) if (TP+FP)>0 else 0
recall    = TP / (TP+FN) if (TP+FN)>0 else 0
f1        = 2*precision*recall/(precision+recall) if (precision+recall)>0 else 0
mcc       = matthews_corrcoef(y_true_ex, y_pred_ex)

print(f'\nMetrics:')
print(f"  Accuracy:  {accuracy:.4f}")
print(f"  Precision: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  F1:        {f1:.4f}")
print(f"  MCC:       {mcc:.4f}  (range -1 to +1, robust to imbalance)")

### Q17: What is cross-validation and when would you not use k-fold?

**Answer:** Cross-validation estimates generalization performance by training and testing on different subsets. Standard k-fold: split into k folds, train on k-1, test on 1, repeat k times, average results.

**When NOT to use standard k-fold:**
- **Time series:** Use time-series split (train on past, test on future)
- **Grouped data:** Use GroupKFold (same patient in train and test = leakage)
- **Very imbalanced:** Use StratifiedKFold (preserves class proportions)
- **Small datasets:** Use leave-one-out (LOOCV) for maximum training data

In [ ]:
from sklearn.model_selection import (KFold, StratifiedKFold, TimeSeriesSplit,
                                      GroupKFold, cross_val_score)
from sklearn.linear_model import LogisticRegression

np.random.seed(42)
X_cv = np.random.randn(200, 5)
y_cv = (X_cv[:, 0] > 0).astype(int)

model_cv = LogisticRegression(random_state=42, max_iter=200)

# Standard KFold
kf = KFold(n_splits=5, shuffle=True, random_state=42)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores_kf  = cross_val_score(model_cv, X_cv, y_cv, cv=kf)
scores_skf = cross_val_score(model_cv, X_cv, y_cv, cv=skf)

print(f"KFold CV:            mean={scores_kf.mean():.4f}, std={scores_kf.std():.4f}")
print(f"Stratified KFold CV: mean={scores_skf.mean():.4f}, std={scores_skf.std():.4f}")

# Time series split demo
tss = TimeSeriesSplit(n_splits=5)
print('\nTimeSeries split (correct for temporal data):')
print('Fold   Train size   Test size')
for fold, (tr, te) in enumerate(tss.split(X_cv), 1):
    print(f"  {fold}      {len(tr):4d}         {len(te):4d}")

print('\nNote: each test fold uses only future data relative to train fold.')
print('Standard k-fold would leak future information into training.')

### Q18: What is the expectation-maximization (EM) algorithm?

**Question (Google, research roles):**

**Answer:** EM is an iterative algorithm for maximum likelihood estimation when data has hidden (latent) variables.

**Two steps per iteration:**
- **E-step (Expectation):** Compute expected values of latent variables given current parameters
- **M-step (Maximization):** Update parameters to maximize expected log-likelihood

**Convergence:** Always increases likelihood (or stays same). May converge to local optimum.

**Used in:** Gaussian Mixture Models, Hidden Markov Models, K-means (hard-assignment EM)

In [ ]:
from sklearn.mixture import GaussianMixture

np.random.seed(42)

# Generate data from mixture of 3 Gaussians
n_per_component = 200
means_true = [[-2, -2], [0, 3], [3, -1]]
covs_true  = [0.5, 0.8, 0.6]

X_parts = []
for mu, cov_scale in zip(means_true, covs_true):
    X_parts.append(np.random.multivariate_normal(
        mu, np.eye(2)*cov_scale, n_per_component))
X_gmm = np.vstack(X_parts)

# Fit GMM with EM
gmm = GaussianMixture(n_components=3, random_state=42, verbose=0)
gmm.fit(X_gmm)

print('GMM fitted with EM algorithm:')
print('True means:')
for mu in means_true:
    print(f'  {mu}')
print('Learned means:')
for mu in gmm.means_:
    print(f"  [{mu[0]:.3f}, {mu[1]:.3f}]")

print(f"\nLog-likelihood: {gmm.score(X_gmm)*len(X_gmm):.2f}")

# K-means as hard-assignment EM
from sklearn.cluster import KMeans
km = KMeans(n_clusters=3, random_state=42, n_init=10)
km.fit(X_gmm)
print('\nK-means centers (hard-assignment EM):')
for c in km.cluster_centers_:
    print(f"  [{c[0]:.3f}, {c[1]:.3f}]")

### Q19: What is batch normalization and why does it help?

**Question (Meta, Google, NVIDIA):**

**Answer:** BatchNorm normalizes layer activations within a mini-batch: x_hat = (x - mu_batch) / sqrt(sigma_batch^2 + epsilon)

Then learns scale and shift: y = gamma * x_hat + beta

**Benefits:**
- Reduces internal covariate shift -- stabilizes distribution of activations
- Allows higher learning rates
- Acts as regularizer (slight noise from batch statistics)
- Reduces sensitivity to weight initialization

**Limitations:** Does not work well with small batches or RNNs (use LayerNorm instead for transformers)

In [ ]:
# Manual batch normalization implementation
def batch_norm_forward(X, gamma, beta, eps=1e-5):
    mu = X.mean(axis=0)
    var = X.var(axis=0)
    X_hat = (X - mu) / np.sqrt(var + eps)
    out = gamma * X_hat + beta
    return out, mu, var, X_hat

np.random.seed(42)

# Simulate activations from a layer
batch_size = 32
n_features = 8

# Poorly conditioned activations
activations = np.random.randn(batch_size, n_features) * 5 + 3

gamma = np.ones(n_features)
beta = np.zeros(n_features)

normed, mu, var, _ = batch_norm_forward(activations, gamma, beta)

print('Batch Normalization Effect:')
print('Feature   Before(mean/std)   After(mean/std)')
for i in range(4):
    bm = activations[:,i].mean(); bs = activations[:,i].std()
    nm = normed[:,i].mean(); ns = normed[:,i].std()
    print(f"  {i}        {bm:.2f}/{bs:.2f}           {nm:.4f}/{ns:.4f}")

print('\nAfter BN: mean~0, std~1 for each feature.')
print('gamma and beta let the network learn optimal scale/shift.')

### Q20: How do transformers use attention, and what is the complexity?

**Question (Google, Meta, OpenAI, Anthropic -- for LLM roles):**

**Answer:** Attention(Q, K, V) = softmax(QK^T / sqrt(d_k)) * V

- Q (query), K (key), V (value) are linear projections of input
- QK^T computes pairwise similarity between all positions
- sqrt(d_k) scaling prevents vanishing gradients in softmax
- Output: weighted sum of values, weighted by attention scores

**Complexity:** O(n^2 * d) where n = sequence length, d = dimension. Quadratic in sequence length -- bottleneck for long contexts.

**Efficient attention variants:** Sparse attention, FlashAttention (IO-aware), linear attention approximations.

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)
    if mask is not None:
        scores = np.where(mask, scores, -1e9)
    # Softmax
    scores_exp = np.exp(scores - scores.max(axis=-1, keepdims=True))
    attn_weights = scores_exp / scores_exp.sum(axis=-1, keepdims=True)
    output = attn_weights @ V
    return output, attn_weights

np.random.seed(42)

# Simple attention example
seq_len = 5
d_model = 8
d_k = 4

# Random input sequence
X_seq = np.random.randn(seq_len, d_model)

# Linear projections
W_Q = np.random.randn(d_model, d_k) * 0.1
W_K = np.random.randn(d_model, d_k) * 0.1
W_V = np.random.randn(d_model, d_k) * 0.1

Q = X_seq @ W_Q
K = X_seq @ W_K
V = X_seq @ W_V

output, attn = scaled_dot_product_attention(Q, K, V)

print(f"Input shape:  {X_seq.shape} (seq_len={seq_len}, d_model={d_model})")
print(f"Q,K,V shape:  {Q.shape} (seq_len={seq_len}, d_k={d_k})")
print(f"Output shape: {output.shape}")

print('\nAttention weights matrix (rows=queries, cols=keys):')
print(np.round(attn, 3))
print(f"\nRow sums (should be 1.0): {attn.sum(axis=1).round(4)}")

# Complexity analysis
print('\nComplexity analysis:')
for n in [64, 256, 1024, 4096]:
    d = 512
    flops = n**2 * d
    print(f"  seq_len={n:5d}: O(n^2*d) = {flops:>12,} ops")

---
## Summary: Key Formulas to Memorize

### Probability
- Bayes: P(A|B) = P(B|A)*P(A) / P(B)
- Chain rule: P(A,B) = P(A|B)*P(B)

### Statistics
- Cohen's d = (mu1 - mu2) / pooled_std
- Bonferroni: use alpha/m per test for m tests

### Information Theory
- Entropy: H(P) = -sum P(x) log P(x)
- KL: KL(P||Q) = sum P(x) log(P(x)/Q(x))
- Cross-entropy: H(P,Q) = H(P) + KL(P||Q)

### Linear Algebra
- Normal equation: w = (X^T X)^-1 X^T y
- SVD: A = U Sigma V^T
- Av = lambda*v (eigenvalue equation)

### Neural Networks
- Softmax: exp(z_i) / sum(exp(z_j))
- Attention: softmax(QK^T / sqrt(d_k)) * V
- Backprop uses chain rule: dL/dW = dL/da * da/dz * dz/dW

In [ ]:
print('Notebook complete!')
print('Topics covered:')
topics = [
    'Section 1: Vectors & Matrices (5 topics)',
    'Section 2: Probability (5 topics)',
    'Section 3: Statistics (5 topics)',
    'Section 4: Information Theory (3 topics)',
    'Section 5: 20 Interview Q&As',
]
for t in topics:
    print(f'  - {t}')